In [1]:
import sys
import os
import argparse

# Instead of __file__, use os.getcwd()
sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

from src.envs import N3il

2.2.5


In [2]:
n=3

current_dir = os.getcwd()

args = {
    'environment': 'N3il',  # Specify the environment
    'algorithm': 'MCTS',
    'max_level_to_use_symmetry': -1,  # Use symmetry for first 2 levels (helps find compact solutions)
    'n': n,
    'C': 1.41,  # 1e-7 for n=20
    'num_searches': 100*(n**2),  # Adjusted for larger n
    'num_workers': 1,      # >1 ⇒ parallel
    'virtual_loss': 1.0,     # magnitude to subtract at reservation
    'process_bar': True,
    'display_state': True,
    'logging_mode': True,  # Enable logging mode to get return value
    'TopN': n,  # Without Priority
    "simulate_with_priority": False,
    'table_dir': current_dir,  # Directory to save tables
    'figure_dir': os.path.join(current_dir, 'figure'),  # Directory to save figures
    'random_seed': 1,  # Use the loop index as a seed for reproducibility
    'tree_visualization': False,  # Set to True to enable tree visualization
}

n3il_test = N3il((n, n), args)

In [10]:
parent_state_test = n3il_test.get_initial_state()
parent_state_test

array([[0, 0, 0],
       [0, 0, 0],
       [0, 0, 0]], dtype=uint8)

In [11]:
parent_state_test = n3il_test.get_next_state(parent_state_test, 7)  # Example action, replace with actual action logic
parent_state_test

array([[0, 0, 0],
       [0, 0, 0],
       [0, 1, 0]], dtype=uint8)

In [12]:
parent_valid_move_test = n3il_test.get_valid_moves(parent_state_test).reshape((n, n))
parent_valid_move_test

array([[1, 1, 1],
       [1, 1, 1],
       [1, 0, 1]], dtype=uint8)

In [13]:
n3il_test.get_valid_moves_subset(parent_state_test, parent_valid_move_test, 1).reshape((n, n))

array([[1, 0, 1],
       [1, 0, 1],
       [1, 0, 1]], dtype=uint8)

In [30]:
parent_state_test = n3il_test.get_next_state(parent_state_test, 3)
parent_state_test

array([[1, 1, 0],
       [1, 0, 0],
       [0, 0, 0]], dtype=uint8)

In [ ]:
n3il_test.get_valid_moves(parent_state_test).reshape((n, n))

array([[0, 0, 0],
       [0, 1, 1],
       [0, 1, 1]], dtype=uint8)

In [37]:
parent_state_test

array([[1, 1, 0],
       [0, 0, 0],
       [0, 0, 0]], dtype=uint8)

In [39]:
parent_valid_move_test

array([[0, 0, 0],
       [1, 1, 1],
       [1, 1, 1]], dtype=uint8)

In [36]:
parent_state_test.reshape(-1)

array([1, 1, 0, 0, 0, 0, 0, 0, 0], dtype=uint8)

In [ ]:
from numba import njit
@njit(cache=True, nogil=True)
def get_valid_moves_subset_nb(parent_state, parent_valid_moves, action_taken, row_count, column_count):
    """
    Given a parent state (2D boolean array) and its valid move mask (1D uint8 array),
    return a refined valid move mask for the child:
      1) Remove the action just taken.
      2) For each existing point in state, compute the line to the new point,
         then invalidate any intermediate grid points that lie exactly on that line.
      3) If slope is infinite, invalidate entire column; if slope is zero, invalidate entire row.
    Returns a flattened uint8 array: 1 = valid, 0 = invalid.
    """
    # Copy input mask and remove the taken action
    mask = parent_valid_moves.copy().reshape(-1)
    mask[action_taken] = 0

    # Coordinates of the newly placed point
    new_r = action_taken // column_count
    new_c = action_taken % column_count

    # Iterate over all existing points
    for pr in range(row_count):
        for pc in range(column_count):
            if not parent_state[pr, pc]:
                continue
            # Skip the new point itself
            if pr == new_r and pc == new_c:
                continue

            dr = pr - new_r
            dc = pc - new_c

            # Infinite slope (vertical line): invalidate entire column
            if dc == 0:
                for rr in range(row_count):
                    idx = rr * column_count + new_c
                    mask[idx] = 0
                continue

            # Zero slope (horizontal line): invalidate entire row
            if dr == 0:
                row_index = pr
                base = row_index * column_count
                for cc in range(column_count):
                    mask[base + cc] = 0
                continue

            # General (non-vertical, non-horizontal) case: remove every point on the infinite line
            # through (new_r,new_c) and (pr,pc), including both the segment and its extensions.
            for cc in range(column_count):
                # compute how far horizontally from the new point
                num = (cc - new_c) * dr
                # only those aligning to integer row are collinear
                if num % dc != 0:
                    continue
                rr = new_r + num // dc
                # skip anything outside the grid
                if rr < 0 or rr >= row_count:
                    continue
                idx = rr * column_count + cc
                mask[idx] = 0

    return mask

In [42]:
get_valid_moves_subset_nb(parent_state_test, parent_valid_move_test, 3, n, n).reshape((n, n))

array([[0, 0, 0],
       [0, 0, 0],
       [1, 1, 1]], dtype=uint8)